# ARTI404 – Image Processing
## Lab 5 – Assessment: Spatial Filtering — Smoothing and Sharpening

| Task | Description |
|------|-------------|
| Task 1 | Convolve an image with a **7×7 box filter**, show original and smoothed side by side |
| Task 2 | Apply a **5×5** and **21×21 Gaussian filter**, show all three images side by side |
| Task 3 | Apply a **3×3 Laplacian filter** to sharpen an image, show original, Laplacian, and sharpened |

## Imports

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage import data

---
## Task 1: 7×7 Box Filter (Mean Smoothing)

A **box filter** (also called a mean filter) replaces each pixel with the average of all pixels in its neighbourhood. A 7×7 kernel averages over 49 pixels, producing a noticeable but uniform blur.

$$h(x,y) = \frac{1}{49} \begin{bmatrix} 1 & \cdots & 1 \\ \vdots & \ddots & \vdots \\ 1 & \cdots & 1 \end{bmatrix}_{7 \times 7}$$

In [ ]:
# ── Load image (using skimage built-in so no external file is needed) ──
image_bgr = cv2.cvtColor(
    (data.astronaut()).astype(np.uint8), cv2.COLOR_RGB2BGR
)
image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)   # display-ready RGB

# ── Build 7×7 box (mean) kernel ──
kernel_size = 7
box_kernel = np.ones((kernel_size, kernel_size), dtype=np.float32) / (kernel_size ** 2)
print('Box kernel (7×7):')
print(np.round(box_kernel, 4))

# ── Apply via cv2.filter2D ──
smoothed_box = cv2.filter2D(image, -1, box_kernel)

# ── Plot ──
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(image)
axes[0].set_title('Original Image', fontsize=13)
axes[0].axis('off')

axes[1].imshow(smoothed_box)
axes[1].set_title('Smoothed — 7×7 Box Filter', fontsize=13)
axes[1].axis('off')

plt.suptitle('Task 1: 7×7 Box Filter', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('task1_box_filter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → task1_box_filter.png')

---
## Task 2: Gaussian Filtering — 5×5 vs 21×21

A **Gaussian filter** weights neighbouring pixels according to a 2-D Gaussian distribution. A larger kernel (and/or larger σ) captures more of the distribution tails → stronger, smoother blur with fewer ringing artefacts than the box filter.

Here we compare:
- **5×5 Gaussian** (mild blur, preserves fine detail)
- **21×21 Gaussian** (heavy blur, strong smoothing)

In [ ]:
# ── Load a different image to demonstrate ──
image2 = cv2.cvtColor(
    (data.camera()).astype(np.uint8), cv2.COLOR_GRAY2RGB
)   # cameraman → RGB for uniform imshow handling

# ── Apply Gaussian filters ──
# sigmaX=0 → OpenCV auto-computes sigma from kernel size: sigma ≈ 0.3*((k-1)*0.5 - 1) + 0.8
gauss_5  = cv2.GaussianBlur(image2, (5,  5),  sigmaX=0)
gauss_21 = cv2.GaussianBlur(image2, (21, 21), sigmaX=0)

print(f'5×5  Gaussian  — auto sigma ≈ {0.3*((5-1)*0.5-1)+0.8:.2f}')
print(f'21×21 Gaussian — auto sigma ≈ {0.3*((21-1)*0.5-1)+0.8:.2f}')

# ── Plot ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(image2, cmap='gray')
axes[0].set_title('Original Image', fontsize=13)
axes[0].axis('off')

axes[1].imshow(gauss_5, cmap='gray')
axes[1].set_title('Gaussian — 5×5 Kernel\n(mild blur)', fontsize=13)
axes[1].axis('off')

axes[2].imshow(gauss_21, cmap='gray')
axes[2].set_title('Gaussian — 21×21 Kernel\n(heavy blur)', fontsize=13)
axes[2].axis('off')

plt.suptitle('Task 2: Gaussian Filtering — Kernel Size Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('task2_gaussian_filter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → task2_gaussian_filter.png')

---
## Task 3: Laplacian Sharpening (3×3 Kernel)

The **Laplacian** is a second-order derivative filter that highlights regions of rapid intensity change (edges). Subtracting it from the original amplifies edges, sharpening the image:

$$g(x,y) = f(x,y) - \nabla^2 f(x,y)$$

Steps:
1. Read image and convert to `float32`
2. Apply `cv2.Laplacian()` with `ksize=3`
3. Sharpen: `sharpened = clip(image_float − laplacian, 0, 1)`
4. Plot original, Laplacian response, and sharpened result

In [ ]:
# ── Load image ──
image3_rgb = cv2.cvtColor(
    (data.coins()).astype(np.uint8), cv2.COLOR_GRAY2RGB
)

# Step 1 – Convert to float32 in [0, 1]
image_float = image3_rgb.astype(np.float32) / 255.0

# Step 2 – Apply 3×3 Laplacian
# cv2.Laplacian returns a float64 array; divide by 255 to normalise scale
laplacian = cv2.Laplacian(image_float, cv2.CV_32F, ksize=3)

# Step 3 – Sharpen: subtract Laplacian from original
sharpened = np.clip(image_float - laplacian, 0, 1)

print(f'Image float range   : [{image_float.min():.3f}, {image_float.max():.3f}]')
print(f'Laplacian range     : [{laplacian.min():.3f},  {laplacian.max():.3f}]')
print(f'Sharpened range     : [{sharpened.min():.3f},  {sharpened.max():.3f}]')

# Step 4 – Plot
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(image_float, cmap='gray')
axes[0].set_title('Original Image', fontsize=13)
axes[0].axis('off')

# Show absolute Laplacian for visibility
axes[1].imshow(np.abs(laplacian), cmap='gray')
axes[1].set_title('Laplacian Response\n(|∇²f|, edge map)', fontsize=13)
axes[1].axis('off')

axes[2].imshow(sharpened, cmap='gray')
axes[2].set_title('Sharpened Image\n(f − ∇²f)', fontsize=13)
axes[2].axis('off')

plt.suptitle('Task 3: Laplacian Sharpening (3×3 Kernel)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('task3_laplacian_sharpening.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → task3_laplacian_sharpening.png')

---
## Summary

| Task | Filter | Effect |
|------|--------|--------|
| 1 | 7×7 Box (Mean) | Uniform averaging; reduces noise but blurs edges |
| 2 | 5×5 Gaussian | Gentle, weighted smoothing; preserves more detail than box |
| 2 | 21×21 Gaussian | Heavy smoothing; strong noise suppression, edges lost |
| 3 | 3×3 Laplacian | Second-order derivative; highlights edges; subtraction sharpens image |

**Key insight:** Smoothing filters (box, Gaussian) suppress high-frequency content (noise, edges). Sharpening with the Laplacian does the opposite — it amplifies high frequencies by penalising pixels that differ from their neighbours.